In [2]:
!git clone https://github.com/SmithC05/amazon-ml-challenge-2026.git /content/amazon-ml-challenge-2026
%cd /content/amazon-ml-challenge-2026
!git rev-parse HEAD

Cloning into '/content/amazon-ml-challenge-2026'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 309 (delta 50), reused 58 (delta 36), pack-reused 235 (from 1)
Receiving objects: 100% (309/309), 316.98 KiB | 5.66 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/amazon-ml-challenge-2026
6e58ffa6471a26e23984115c456d1f1c13b82b40


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

cache = Path("/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache")

for f in ["train_source1.parquet", "train_source2.parquet", "train_source3.parquet"]:
    print(f, "✅" if (cache / f).exists() else "❌")

train_source1.parquet ✅
train_source2.parquet ✅
train_source3.parquet ✅


RAM: 13.61 GB
CPU: 2


In [7]:
!python /content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py \
--cache-dir "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache" \
--gt "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv" \
--output-dir "/content/stage2_eval" \
--n-s1 10000 \
--max-df 500 1000


STAGE 2 M4 EVALUATION (No Fuzzy)
S1 rows : 10,000

--- Running block: exact ---
Done in 4.8s. Recall: 24.66%, Pairs: 109,561

--- Running block: address ---
KeyboardInterrupt

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 337, in <module>
    main()
    ~~~~^^
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 189, in main
    generate_candidates_memory_safe(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        split="train",
        ^^^^^^^^^^^^^^
    ...<6 lines>...
        db_path=out_dir / "m4_work.duckdb"
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/amazon-ml-challenge-2026/src/candidate_generation.py", line 1120, in generate_candidates_memory_safe
    con.execute(f"""
    ~~~~~~~~~~~^^^^^
        COPY (
        ^^^^^^
    ...<20 lines>...
        ) TO '{out_file_str}' (FORMAT CSV, DELIMITER '\t

In [8]:
%cd /content/amazon-ml-challenge-2026
!git pull origin main

/content/amazon-ml-challenge-2026
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.26 KiB | 143.00 KiB/s, done.
From https://github.com/SmithC05/amazon-ml-challenge-2026
 * branch            main       -> FETCH_HEAD
   6e58ffa..c109d21  main       -> origin/main
Updating 6e58ffa..c109d21
Fast-forward
 src/candidate_generation.py     | 43 ++++++++++++++++++++++++++++++++++++++--
 tools/evaluate_stage2_blocks.py | 44 +++++++++++++++++++++++++++++++++++++++++
 2 files changed, 85 insertions(+), 2 deletions(-)


In [9]:
!python tools/evaluate_stage2_blocks.py \
--cache-dir "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache" \
--gt "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv" \
--output-dir "/content/address_eval" \
--n-s1 10000 \
--address-max-df 50 100 500

STAGE 2 M4 EVALUATION (No Fuzzy)
S1 rows : 10,000

--- Running block: exact ---
Done in 9.6s. Recall: 24.66%, Pairs: 109,561

--- Running block: address ---
Done in 57.3s. Recall: 11.83%, Pairs: 124,422

--- Running block: prefix ---
Traceback (most recent call last):
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 381, in <module>
    main()
    ~~~~^^
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 203, in main
    metrics = _evaluate_tsv(out_tsv, truth, s1_ids)
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 98, in _evaluate_tsv
    for row in reader:
               ^^^^^^
  File "/usr/lib/python3.13/csv.py", line 178, in __next__
    row = next(self.reader)
_csv.Error: field larger than field limit (131072)


In [10]:
from pathlib import Path

p = Path("/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py")

text = p.read_text()

needle = "import csv\n"
replacement = "import csv\nimport sys\ncsv.field_size_limit(sys.maxsize)\n"

if "csv.field_size_limit" not in text:
    text = text.replace(needle, replacement, 1)
    p.write_text(text)
    print("✅ Patched CSV field-size limit")
else:
    print("✅ CSV field-size limit already patched")

✅ Patched CSV field-size limit


In [11]:
!grep -n "field_size_limit" tools/evaluate_stage2_blocks.py

28:csv.field_size_limit(sys.maxsize)


In [13]:
!python tools/evaluate_stage2_blocks.py \
--cache-dir "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache" \
--gt "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv" \
--output-dir "/content/address_eval" \
--n-s1 10000 \
--address-max-df 50 100 500

STAGE 2 M4 EVALUATION (No Fuzzy)
S1 rows : 10,000

--- Running block: exact ---
Done in 4.2s. Recall: 24.66%, Pairs: 109,561

--- Running block: address ---
Done in 57.3s. Recall: 11.83%, Pairs: 124,422

--- Running block: prefix ---
Done in 235.9s. Recall: 76.57%, Pairs: 91,694,800

--- Running block: country_token ---
  Block [country_token] x S3 FAILED: Query interrupted
KeyboardInterrupt

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/amazon-ml-challenge-2026/src/candidate_generation.py", line 1023, in generate_candidates_memory_safe
    con.execute(query)
    ~~~~~~~~~~~^^^^^^^
RuntimeError: Query interrupted
^C


In [14]:
!python tools/evaluate_stage2_blocks.py --help

usage: evaluate_stage2_blocks.py [-h] --cache-dir CACHE_DIR --gt GT
                                 --output-dir OUTPUT_DIR [--n-s1 N_S1]
                                 [--max-df MAX_DF [MAX_DF ...]]
                                 [--address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]]

Stage 2 M4 blocking evaluation.

options:
  -h, --help            show this help message and exit
  --cache-dir CACHE_DIR
                        Path to M2 cache containing Parquet files
  --gt GT               Path to train_ground_truth.tsv
  --output-dir OUTPUT_DIR
                        Directory to save intermediate and union TSVs
  --n-s1 N_S1           Number of S1 rows to evaluate
  --max-df MAX_DF [MAX_DF ...]
                        token_max_df thresholds to sweep
  --address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]
                        address_max_df thresholds to sweep


In [15]:
%cd /content/amazon-ml-challenge-2026
!git status --short
!grep -n "prefix_max_df" src/candidate_generation.py
!grep -n "prefix-max-df" tools/evaluate_stage2_blocks.py

/content/amazon-ml-challenge-2026
 M tools/evaluate_stage2_blocks.py


In [16]:
!python tools/evaluate_stage2_blocks.py \
--cache-dir "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache" \
--gt "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv" \
--output-dir "/content/prefix_eval" \
--n-s1 10000 \
--prefix-max-df 100 500 1000 5000

usage: evaluate_stage2_blocks.py [-h] --cache-dir CACHE_DIR --gt GT
                                 --output-dir OUTPUT_DIR [--n-s1 N_S1]
                                 [--max-df MAX_DF [MAX_DF ...]]
                                 [--address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]]
evaluate_stage2_blocks.py: error: unrecognized arguments: --prefix-max-df 100 500 1000 5000


In [17]:
%cd /content/amazon-ml-challenge-2026

!grep -n "prefix-max-df" tools/evaluate_stage2_blocks.py
!grep -n "prefix_max_df" src/candidate_generation.py
!git status
!git log -1 --oneline

/content/amazon-ml-challenge-2026
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   tools/evaluate_stage2_blocks.py

no changes added to commit (use "git add" and/or "git commit -a")
c109d21 (HEAD -> main, origin/main, origin/HEAD) feat: add frequency-aware address blocking and evaluation sweeps


In [20]:
!grep -n "prefix_max_df" src/candidate_generation.py
!grep -n "prefix-max-df" tools/evaluate_stage2_blocks.py

In [21]:
%cd /content/amazon-ml-challenge-2026

!python -m py_compile src/candidate_generation.py tools/evaluate_stage2_blocks.py
!python tools/evaluate_stage2_blocks.py --help

/content/amazon-ml-challenge-2026
usage: evaluate_stage2_blocks.py [-h] --cache-dir CACHE_DIR --gt GT
                                 --output-dir OUTPUT_DIR [--n-s1 N_S1]
                                 [--max-df MAX_DF [MAX_DF ...]]
                                 [--address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]]

Stage 2 M4 blocking evaluation.

options:
  -h, --help            show this help message and exit
  --cache-dir CACHE_DIR
                        Path to M2 cache containing Parquet files
  --gt GT               Path to train_ground_truth.tsv
  --output-dir OUTPUT_DIR
                        Directory to save intermediate and union TSVs
  --n-s1 N_S1           Number of S1 rows to evaluate
  --max-df MAX_DF [MAX_DF ...]
                        token_max_df thresholds to sweep
  --address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]
                        address_max_df thresholds to sweep


In [23]:
%cd /content/amazon-ml-challenge-2026

!git restore tools/evaluate_stage2_blocks.py
!git pull origin main
!git log -1 --oneline

/content/amazon-ml-challenge-2026
From https://github.com/SmithC05/amazon-ml-challenge-2026
 * branch            main       -> FETCH_HEAD
Updating c109d21..b91f1cb
Fast-forward
 src/candidate_generation.py     | 67 ++++++++++++++++++++++++++++++++++++-----
 tools/evaluate_stage2_blocks.py | 46 ++++++++++++++++++++++++++++
 2 files changed, 105 insertions(+), 8 deletions(-)
b91f1cb (HEAD -> main, origin/main, origin/HEAD) feat: add frequency-aware prefix blocking


In [24]:
!python tools/evaluate_stage2_blocks.py --help

usage: evaluate_stage2_blocks.py [-h] --cache-dir CACHE_DIR --gt GT
                                 --output-dir OUTPUT_DIR [--n-s1 N_S1]
                                 [--max-df MAX_DF [MAX_DF ...]]
                                 [--address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]]
                                 [--prefix-max-df PREFIX_MAX_DF [PREFIX_MAX_DF ...]]

Stage 2 M4 blocking evaluation.

options:
  -h, --help            show this help message and exit
  --cache-dir CACHE_DIR
                        Path to M2 cache containing Parquet files
  --gt GT               Path to train_ground_truth.tsv
  --output-dir OUTPUT_DIR
                        Directory to save intermediate and union TSVs
  --n-s1 N_S1           Number of S1 rows to evaluate
  --max-df MAX_DF [MAX_DF ...]
                        token_max_df thresholds to sweep
  --address-max-df ADDRESS_MAX_DF [ADDRESS_MAX_DF ...]
                        address_max_df thresholds to sweep
  --prefix-max-df PREFIX_MAX

In [27]:
!python tools/evaluate_stage2_blocks.py \
--cache-dir "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache" \
--gt "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv" \
--output-dir "/content/prefix_eval" \
--n-s1 10000 \
--prefix-max-df 100 500 1000

STAGE 2 M4 EVALUATION (No Fuzzy)
S1 rows : 10,000

--- PREFIX THRESHOLD BENCHMARK ---

--- Running block: prefix_100 ---
KeyboardInterrupt

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 427, in <module>
    main()
    ~~~~^^
  File "/content/amazon-ml-challenge-2026/tools/evaluate_stage2_blocks.py", line 192, in main
    generate_candidates_memory_safe(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        split="train",
        ^^^^^^^^^^^^^^
    ...<7 lines>...
        db_path=out_dir / "m4_work.duckdb"
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/amazon-ml-challenge-2026/src/candidate_generation.py", line 865, in generate_candidates_memory_safe
    con.execute("""
    ~~~~~~~~~~~^^^^
        CREATE OR REPLACE TABLE rhs_prefix_df AS
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
        GROUP BY prefix
    

In [26]:
!echo "=== ADDRESS FILES ==="
!find /content/address_eval -maxdepth 1 -type f -printf "%f %k KB\n" 2>/dev/null | sort

!echo ""
!echo "=== PREFIX FILES ==="
!find /content/prefix_eval -maxdepth 1 -type f -printf "%f %k KB\n" 2>/dev/null | sort

=== ADDRESS FILES ===
block_address.tsv 1720 KB
block_country_token.tsv 4 KB
block_exact.tsv 1516 KB
block_prefix.tsv 1154296 KB
m4_work.duckdb 240144 KB

=== PREFIX FILES ===
block_prefix_1000.tsv 26664 KB
block_prefix_100.tsv 1776 KB
block_prefix_500.tsv 12264 KB


In [31]:
import duckdb
from pathlib import Path

base = Path("/content")
out = Path("/content/union_10k")
out.mkdir(exist_ok=True)

files = [
    base / "address_eval/block_exact.tsv",
    base / "address_eval/block_address.tsv",
    base / "prefix_eval/block_prefix_1000.tsv",
]

queries = []

for f in files:
    p = str(f).replace("\\", "/")
    queries.append(f"""
        SELECT
            source1_entity_id,
            unnest(string_split(candidate_entity_ids, ',')) AS candidate_entity_id
        FROM read_csv_auto('{p}', delim='\\t', header=True)
        WHERE candidate_entity_ids IS NOT NULL
          AND candidate_entity_ids != ''
    """)

query = " UNION ALL ".join(queries)

con = duckdb.connect(str(out / "union.duckdb"))
con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA preserve_insertion_order=false")

tmp_dir = str(out).replace("\\", "/")
con.execute(f"PRAGMA temp_directory='{tmp_dir}'")

union_path = out / "candidate_pairs_10k.tsv"
union_path_str = str(union_path).replace("\\", "/")

s1_parquet = (
    "/content/drive/MyDrive/Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache/train_source1.parquet"
)

con.execute(f"""
COPY (
    WITH all_pairs AS (
        {query}
    ),
    distinct_pairs AS (
        SELECT DISTINCT
            source1_entity_id,
            candidate_entity_id
        FROM all_pairs
    ),
    aggregated AS (
        SELECT
            source1_entity_id,
            string_agg(candidate_entity_id, ',') AS candidate_entity_ids
        FROM distinct_pairs
        GROUP BY source1_entity_id
    ),
    s1_all AS (
        SELECT entity_id AS source1_entity_id
        FROM read_parquet('{s1_parquet}')
        LIMIT 10000
    )
    SELECT
        s1_all.source1_entity_id,
        COALESCE(
            aggregated.candidate_entity_ids,
            ''
        ) AS candidate_entity_ids
    FROM s1_all
    LEFT JOIN aggregated
      ON s1_all.source1_entity_id = aggregated.source1_entity_id
) TO '{union_path_str}'
(FORMAT CSV, DELIMITER '\\t', HEADER)
""")

con.close()

print("✅ Union created:")
print(union_path)

✅ Union created:
/content/union_10k/candidate_pairs_10k.tsv


In [32]:
import csv
import pandas as pd

csv.field_size_limit(10**9)

gt_path = (
    "/content/drive/MyDrive/Amazon ML Challenge 2026/"
    "01_Dataset/ raw/train/train_ground_truth.tsv"
)

candidate_path = "/content/union_10k/candidate_pairs_10k.tsv"

gt = pd.read_csv(gt_path, sep="\t")

truth = {}
for _, row in gt.iterrows():
    val = row["matched_entity_ids"]
    truth[row["source1_entity_id"]] = (
        {x.strip() for x in str(val).split(",") if x.strip()}
        if pd.notna(val) and str(val).strip()
        else set()
    )

found = 0
lost = 0
total_candidates = 0
covered = 0
max_candidates = 0
n_rows = 0

with open(candidate_path, encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        n_rows += 1

        candidate_ids = {
            x.strip()
            for x in (row["candidate_entity_ids"] or "").split(",")
            if x.strip()
        }

        total_candidates += len(candidate_ids)
        max_candidates = max(max_candidates, len(candidate_ids))

        if candidate_ids:
            covered += 1

        true_ids = truth.get(row["source1_entity_id"], set())

        found += len(true_ids & candidate_ids)
        lost += len(true_ids - candidate_ids)

total_true = found + lost

print("=" * 55)
print("10K UNION EVALUATION")
print("=" * 55)
print(f"S1 rows              : {n_rows:,}")
print(f"Total candidate IDs  : {total_candidates:,}")
print(f"Average candidates   : {total_candidates / n_rows:.1f}")
print(f"Maximum candidates   : {max_candidates:,}")
print(f"S1 with candidates   : {covered:,}")
print(f"True matches found   : {found:,}")
print(f"True matches lost    : {lost:,}")
print(f"Candidate recall     : {found / total_true:.2%}")
print("=" * 55)

10K UNION EVALUATION
S1 rows              : 10,000
Total candidate IDs  : 2,328,511
Average candidates   : 232.9
Maximum candidates   : 2,099
S1 with candidates   : 8,998
True matches found   : 18,480
True matches lost    : 16,272
Candidate recall     : 53.18%


In [34]:
from pathlib import Path
import sys

sys.path.insert(0, "/content/amazon-ml-challenge-2026/src")

from candidate_generation import generate_candidates_memory_safe

cache_dir = Path(
    "/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache"
)

out_dir = Path("/content/token_eval")
out_dir.mkdir(exist_ok=True)

for df in [500, 1000]:
    out_file = out_dir / f"block_token_{df}.tsv"

    print(f"\n{'='*60}")
    print(f"GENERATING TOKEN BLOCK df={df}")
    print(f"{'='*60}")

    generate_candidates_memory_safe(
        split="train",
        cache_dir=cache_dir,
        out_file=out_file,
        blocks=["token"],
        verbose=True,
        chunk_size=10_000,
        threads=2,
        token_max_df=df,
        db_path=out_dir / "m4_work.duckdb",
    )

    print(f"✅ Created: {out_file}")



GENERATING TOKEN BLOCK df=500
Opening file-backed DuckDB at /content/token_eval/m4_work.duckdb
  memory_limit=6GB  threads=2
  temp_directory=/content/token_eval/m4_duckdb_tmp

Creating narrow S1 view over train_source1.parquet (2,206,821 rows)...

Processing S2 (source2)...
  Building RHS relations for S2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Token index stats (max_df=500): total=788,624  usable=786,544  max_df=193,696  median_df=1.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  RHS relations built in 21.8s
    source=S2 block=token chunk=1/221 RAM=2.45GB
      -> chunk 1: 577,277 pairs (total so far 577,277, 0.6s)
    source=S2 block=token chunk=2/221 RAM=2.44GB
      -> chunk 2: 562,701 pairs (total so far 1,139,978, 1.6s)
    source=S2 block=token chunk=3/221 RAM=2.44GB
      -> chunk 3: 558,258 pairs (total so far 1,698,236, 2.2s)
    source=S2 block=token chunk=4/221 RAM=2.44GB
      -> chunk 4: 543,858 pairs (total so far 2,242,094, 2.8s)
    source=S2 block=token chunk=5/221 RAM=2.44GB
      -> chunk 5: 570,001 pairs (total so far 2,812,095, 3.7s)
    source=S2 block=token chunk=6/221 RAM=2.44GB
  Block [token] x S2 FAILED: Query interrupted


KeyboardInterrupt

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/amazon-ml-challenge-2026/src/candidate_generation.py", line 1021, in generate_candidates_memory_safe
    con.execute(query)
    ~~~~~~~~~~~^^^^^^^
RuntimeError: Query interrupted



Processing S3 (source3)...
  Building RHS relations for S3...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Error: KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /usr/local/lib/python3.13/dist-packages/traitlets/traitlets.py(720): __set__
  /content/amazon-ml-challenge-2026/src/candidate_generation.py(762): generate_candidates_memory_safe
  /tmp/ipykernel_2363/187195621.py(22): <cell line: 0>
  /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py(3553): run_code
  /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py(3473): run_ast_nodes
  /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py(3257): run_cell_async
  /usr/local/lib/python3.13/dist-packages/IPython/core/async_helpers.py(78): _pseudo_sync_runner
  /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py(3030): _run_cell
  /usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py(2975): run_cell
  /usr/local/lib/python3.13/dist-packages/ipykernel/zmqshell.py(528): run_cell
  /usr/local/lib/python3.13/dist-packages/ipykernel/ipkernel.py(383): do_e

In [1]:

from pathlib import Path
import duckdb

full_cache = Path(
    "/content/drive/MyDrive/Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache"
)

mini_cache = Path("/content/token_10k_cache")
mini_cache.mkdir(exist_ok=True)

db = duckdb.connect()

src1 = str(full_cache / "train_source1.parquet").replace("\\", "/")
dst1 = str(mini_cache / "train_source1.parquet").replace("\\", "/")

db.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{src1}')
        LIMIT 10000
    ) TO '{dst1}' (FORMAT PARQUET)
""")

db.close()

# Link the full S2/S3 caches
for src in ["train_source2.parquet", "train_source3.parquet"]:
    dst = mini_cache / src
    if not dst.exists():
        dst.symlink_to(full_cache / src)

print("✅ 10K mini-cache ready")
print("S1:", mini_cache / "train_source1.parquet")
print("S2:", mini_cache / "train_source2.parquet")
print("S3:", mini_cache / "train_source3.parquet")

✅ 10K mini-cache ready
S1: /content/token_10k_cache/train_source1.parquet
S2: /content/token_10k_cache/train_source2.parquet
S3: /content/token_10k_cache/train_source3.parquet


In [2]:
from pathlib import Path
import sys

sys.path.insert(0, "/content/amazon-ml-challenge-2026/src")

from candidate_generation import generate_candidates_memory_safe

cache_dir = Path("/content/token_10k_cache")
out_dir = Path("/content/token_10k")
out_dir.mkdir(exist_ok=True)

for df in [500, 1000]:
    print("\n" + "=" * 60)
    print(f"TOKEN BLOCK — df={df}")
    print("=" * 60)

    generate_candidates_memory_safe(
        split="train",
        cache_dir=cache_dir,
        out_file=out_dir / f"block_token_{df}.tsv",
        blocks=["token"],
        verbose=True,
        chunk_size=10_000,
        threads=2,
        token_max_df=df,
        db_path=out_dir / "m4_work.duckdb",
    )

    print(f"✅ Finished token df={df}")


TOKEN BLOCK — df=500
Opening file-backed DuckDB at /content/token_10k/m4_work.duckdb
  memory_limit=6GB  threads=2
  temp_directory=/content/token_10k/m4_duckdb_tmp

Creating narrow S1 view over train_source1.parquet (10,000 rows)...

Processing S2 (source2)...
  Building RHS relations for S2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Token index stats (max_df=500): total=788,624  usable=786,544  max_df=193,696  median_df=1.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  RHS relations built in 18.2s
    source=S2 block=token chunk=1/1 RAM=0.35GB
      -> chunk 1: 577,277 pairs (total so far 577,277, 0.6s)
  -> token x S2:  577,277 pairs TOTAL (0.6s)

Processing S3 (source3)...
  Building RHS relations for S3...


...

  Token index stats (max_df=1000): total=841,153  usable=840,054  max_df=220,809  median_df=1.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  RHS relations built in 37.0s
    source=S3 block=token chunk=1/1 RAM=0.38GB
      -> chunk 1: 2,228,985 pairs (total so far 2,228,985, 1.7s)
  -> token x S3: 2,228,985 pairs TOTAL (1.7s)

Merging 2 block files with DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Merged in 2.6s

  S1 entities with >=1 candidate: 5,342 / 10,000
✅ Finished token df=1000


In [5]:
from pathlib import Path

for f in Path("/content/token_10k").glob("*"):
    print(f)

/content/token_10k/m4_duckdb_tmp
/content/token_10k/block_token_1000.tsv
/content/token_10k/block_token_500.tsv


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pathlib import Path
import shutil

src_dir = Path("/content/token_10k")

dst_dir = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/token_10k"
)

dst_dir.mkdir(parents=True, exist_ok=True)

for name in ["block_token_500.tsv", "block_token_1000.tsv"]:
    src = src_dir / name
    dst = dst_dir / name

    shutil.copy2(src, dst)
    print(f"✅ Saved: {dst}")

print("\nDone.")

✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/token_10k/block_token_500.tsv
✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/token_10k/block_token_1000.tsv

Done.


In [8]:
from pathlib import Path

for p in [
    "/content/address_eval/block_exact.tsv",
    "/content/address_eval/block_address.tsv",
    "/content/prefix_eval/block_prefix_1000.tsv",
    "/content/token_10k/block_token_500.tsv",
]:
    f = Path(p)
    print(("✅" if f.exists() else "❌"), f)

✅ /content/address_eval/block_exact.tsv
✅ /content/address_eval/block_address.tsv
✅ /content/prefix_eval/block_prefix_1000.tsv
✅ /content/token_10k/block_token_500.tsv


In [9]:
# ============================================================
# 10K UNION: exact + address + prefix_1000 + token_500
# ============================================================

from pathlib import Path
import duckdb
import csv
import pandas as pd
import shutil

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
block_files = [
    Path("/content/address_eval/block_exact.tsv"),
    Path("/content/address_eval/block_address.tsv"),
    Path("/content/prefix_eval/block_prefix_1000.tsv"),
    Path("/content/token_10k/block_token_500.tsv"),
]

for f in block_files:
    if not f.exists():
        raise FileNotFoundError(f"Missing block file: {f}")

union_dir = Path("/content/union_10k")
union_dir.mkdir(exist_ok=True)

union_path = union_dir / "candidate_pairs_10k.tsv"
db_path = union_dir / "union.duckdb"

# Existing M2 cache in Drive
cache_dir = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache"
)

s1_parquet = cache_dir / "train_source1.parquet"

if not s1_parquet.exists():
    raise FileNotFoundError(f"Missing S1 parquet: {s1_parquet}")

# ------------------------------------------------------------
# Build DuckDB UNION
# ------------------------------------------------------------
queries = []

for f in block_files:
    p = str(f.resolve()).replace("\\", "/")

    queries.append(f"""
        SELECT
            source1_entity_id,
            unnest(string_split(candidate_entity_ids, ',')) AS candidate_entity_id
        FROM read_csv_auto(
            '{p}',
            delim='\\t',
            header=True
        )
        WHERE candidate_entity_ids IS NOT NULL
          AND trim(candidate_entity_ids) != ''
    """)

all_pairs_query = "\nUNION ALL\n".join(queries)

con = duckdb.connect(str(db_path))

con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute(
    "PRAGMA temp_directory='" +
    str(union_dir.resolve()).replace("\\", "/") +
    "'"
)

s1_path = str(s1_parquet.resolve()).replace("\\", "/")
out_path = str(union_path.resolve()).replace("\\", "/")

print("=" * 60)
print("BUILDING 10K UNION")
print("=" * 60)

con.execute(f"""
COPY (
    WITH all_pairs AS (
        {all_pairs_query}
    ),

    distinct_pairs AS (
        SELECT DISTINCT
            source1_entity_id,
            candidate_entity_id
        FROM all_pairs
        WHERE candidate_entity_id IS NOT NULL
          AND trim(candidate_entity_id) != ''
    ),

    aggregated AS (
        SELECT
            source1_entity_id,
            string_agg(
                candidate_entity_id,
                ',' ORDER BY candidate_entity_id
            ) AS candidate_entity_ids
        FROM distinct_pairs
        GROUP BY source1_entity_id
    ),

    s1_all AS (
        SELECT entity_id AS source1_entity_id
        FROM read_parquet('{s1_path}')
        LIMIT 10000
    )

    SELECT
        s1_all.source1_entity_id,
        COALESCE(aggregated.candidate_entity_ids, '') AS candidate_entity_ids

    FROM s1_all

    LEFT JOIN aggregated
        ON s1_all.source1_entity_id = aggregated.source1_entity_id

) TO '{out_path}'
(FORMAT CSV, DELIMITER '\\t', HEADER)
""")

con.close()

print("\n✅ Union created:")
print(union_path)
print("Size:",
      round(union_path.stat().st_size / (1024**2), 2),
      "MB")

# ------------------------------------------------------------
# Copy union to Drive immediately
# ------------------------------------------------------------
drive_union_dir = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/union_10k"
)

drive_union_dir.mkdir(parents=True, exist_ok=True)

drive_union_path = drive_union_dir / union_path.name
shutil.copy2(union_path, drive_union_path)

print("\n✅ Saved permanently to Drive:")
print(drive_union_path)

# ------------------------------------------------------------
# Locate ground truth automatically
# ------------------------------------------------------------
possible_gt = [
    Path(
        "/content/drive/MyDrive/"
        "Amazon ML Challenge 2026/"
        "01_Dataset/raw/train/train_ground_truth.tsv"
    ),
    Path(
        "/content/drive/MyDrive/"
        "Amazon ML Challenge 2026/"
        "01_Dataset/ raw/train/train_ground_truth.tsv"
    ),
]

gt_path = next((p for p in possible_gt if p.exists()), None)

if gt_path is None:
    # Search only inside the challenge folder
    roots = [
        Path(
            "/content/drive/MyDrive/"
            "Amazon ML Challenge 2026"
        )
    ]

    found = []
    for root in roots:
        if root.exists():
            found.extend(root.rglob("train_ground_truth.tsv"))

    if found:
        gt_path = found[0]

if gt_path is None:
    raise FileNotFoundError(
        "Could not locate train_ground_truth.tsv inside "
        "'Amazon ML Challenge 2026'."
    )

print("\n✅ Ground truth:")
print(gt_path)

# ------------------------------------------------------------
# Evaluate candidate recall
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("10K UNION RECALL")
print("=" * 60)

csv.field_size_limit(10**9)

gt = pd.read_csv(gt_path, sep="\t")

truth = {}

for _, row in gt.iterrows():
    value = row["matched_entity_ids"]

    if pd.isna(value) or not str(value).strip():
        truth[row["source1_entity_id"]] = set()
    else:
        truth[row["source1_entity_id"]] = {
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        }

found = 0
lost = 0
total_candidates = 0
covered = 0
max_candidates = 0
n_rows = 0

with union_path.open(
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        n_rows += 1

        candidate_ids = {
            x.strip()
            for x in (row["candidate_entity_ids"] or "").split(",")
            if x.strip()
        }

        total_candidates += len(candidate_ids)

        if candidate_ids:
            covered += 1

        max_candidates = max(
            max_candidates,
            len(candidate_ids)
        )

        true_ids = truth.get(
            row["source1_entity_id"],
            set()
        )

        found += len(true_ids & candidate_ids)
        lost += len(true_ids - candidate_ids)

total_true = found + lost

print(f"S1 rows             : {n_rows:,}")
print(f"Total candidates    : {total_candidates:,}")
print(f"Average candidates  : {total_candidates / n_rows:.1f}")
print(f"Maximum candidates  : {max_candidates:,}")
print(f"S1 with candidates  : {covered:,}")
print(f"True matches found  : {found:,}")
print(f"True matches lost   : {lost:,}")
print(f"Candidate recall    : {found / total_true:.2%}")

print("=" * 60)

BUILDING 10K UNION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Union created:
/content/union_10k/candidate_pairs_10k.tsv
Size: 38.6 MB

✅ Saved permanently to Drive:
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/union_10k/candidate_pairs_10k.tsv

✅ Ground truth:
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv

10K UNION RECALL
S1 rows             : 10,000
Total candidates    : 3,130,620
Average candidates  : 313.1
Maximum candidates  : 3,128
S1 with candidates  : 9,333
True matches found  : 21,749
True matches lost   : 13,003
Candidate recall    : 62.58%


In [10]:
# ============================================================
# M3 SANITY TEST — USE THE EXISTING 10K UNION
# ============================================================

from pathlib import Path
import csv
import pandas as pd

# Existing files
CANDIDATES = Path("/content/union_10k/candidate_pairs_10k.tsv")

GT_PATH = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/ raw/train/train_ground_truth.tsv"
)

CACHE_DIR = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache"
)

print("Candidates :", CANDIDATES)
print("Ground truth:", GT_PATH)
print("Cache       :", CACHE_DIR)

print("\nExistence checks:")
print("Candidates :", CANDIDATES.exists())
print("GT         :", GT_PATH.exists())
print("S1 cache   :", (CACHE_DIR / "train_source1.parquet").exists())
print("S2 cache   :", (CACHE_DIR / "train_source2.parquet").exists())
print("S3 cache   :", (CACHE_DIR / "train_source3.parquet").exists())

Candidates : /content/union_10k/candidate_pairs_10k.tsv
Ground truth: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv
Cache       : /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache

Existence checks:
Candidates : True
GT         : True
S1 cache   : True
S2 cache   : True
S3 cache   : True


In [11]:
# ============================================================
# M3 SANITY TEST — 1,000 S1 ENTITIES
# Uses existing 10K UNION
# ============================================================

from pathlib import Path
import csv
import shutil
import duckdb
import pandas as pd
import sys

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
REPO = Path("/content/amazon-ml-challenge-2026")
SRC = REPO / "src"

UNION = Path("/content/union_10k/candidate_pairs_10k.tsv")

FULL_CACHE = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache"
)

# Find ground truth
GT_CANDIDATES = [
    Path(
        "/content/drive/MyDrive/"
        "Amazon ML Challenge 2026/"
        "01_Dataset/raw/train/train_ground_truth.tsv"
    ),
    Path(
        "/content/drive/MyDrive/"
        "Amazon ML Challenge 2026/"
        "01_Dataset/ raw/train/train_ground_truth.tsv"
    ),
]

GT_FULL = next((p for p in GT_CANDIDATES if p.exists()), None)

if GT_FULL is None:
    found = list(
        Path("/content/drive/MyDrive/Amazon ML Challenge 2026")
        .rglob("train_ground_truth.tsv")
    )
    if not found:
        raise FileNotFoundError("train_ground_truth.tsv not found.")
    GT_FULL = found[0]

# ------------------------------------------------------------
# Check inputs
# ------------------------------------------------------------
for p, label in [
    (REPO, "Repository"),
    (UNION, "10K union"),
    (FULL_CACHE, "M2 cache"),
    (GT_FULL, "Ground truth"),
]:
    print(f"{'✅' if p.exists() else '❌'} {label}: {p}")

# ------------------------------------------------------------
# Create sandbox folders
# ------------------------------------------------------------
BASE = Path("/content/m3_sanity")
DATA_DIR = BASE / "data"
CACHE_DIR = BASE / "cache"
OUTPUT_DIR = BASE / "output"
MODELS_DIR = BASE / "models"

for d in [DATA_DIR, CACHE_DIR, OUTPUT_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Select first 1,000 S1 rows from existing union
# ------------------------------------------------------------
MINI_CAND = BASE / "candidate_pairs_1000.tsv"

print("\nSelecting first 1,000 S1 entities...")

with UNION.open("r", encoding="utf-8", newline="") as fin:
    reader = csv.DictReader(fin, delimiter="\t")

    rows = []
    for row in reader:
        rows.append(row)
        if len(rows) >= 1000:
            break

if len(rows) < 1000:
    raise RuntimeError(
        f"Union contains only {len(rows)} rows; expected 1,000+."
    )

with MINI_CAND.open("w", encoding="utf-8", newline="") as fout:
    writer = csv.DictWriter(
        fout,
        fieldnames=["source1_entity_id", "candidate_entity_ids"],
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ Created: {MINI_CAND}")

# ------------------------------------------------------------
# 2. Collect candidate IDs by source
# ------------------------------------------------------------
s1_ids = set()
s2_ids = set()
s3_ids = set()

for row in rows:
    s1_ids.add(row["source1_entity_id"])

    value = row.get("candidate_entity_ids", "") or ""

    for cid in value.split(","):
        cid = cid.strip()

        if not cid:
            continue

        if cid.startswith("S2-"):
            s2_ids.add(cid)
        elif cid.startswith("S3-"):
            s3_ids.add(cid)

print("\nCandidate ID counts:")
print("S1:", len(s1_ids))
print("S2:", len(s2_ids))
print("S3:", len(s3_ids))

# ------------------------------------------------------------
# 3. Create filtered M2 cache using DuckDB
# ------------------------------------------------------------
db = duckdb.connect()

db.execute("PRAGMA memory_limit='6GB'")
db.execute("PRAGMA threads=2")
db.execute("PRAGMA preserve_insertion_order=false")

print("\nCreating mini M2 cache...")

# S1
s1_df = pd.DataFrame(
    {"entity_id": sorted(s1_ids)}
)

db.register("mini_s1_ids", s1_df)

db.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source1.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN mini_s1_ids i
            ON s.entity_id = i.entity_id
    )
    TO '{str(CACHE_DIR / "train_source1.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

# S2
s2_df = pd.DataFrame(
    {"entity_id": sorted(s2_ids)}
)

db.register("mini_s2_ids", s2_df)

db.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source2.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN mini_s2_ids i
            ON s.entity_id = i.entity_id
    )
    TO '{str(CACHE_DIR / "train_source2.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

# S3
s3_df = pd.DataFrame(
    {"entity_id": sorted(s3_ids)}
)

db.register("mini_s3_ids", s3_df)

db.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source3.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN mini_s3_ids i
            ON s.entity_id = i.entity_id
    )
    TO '{str(CACHE_DIR / "train_source3.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

db.close()

print("✅ Mini cache created")

# ------------------------------------------------------------
# 4. Verify mini cache sizes
# ------------------------------------------------------------
db = duckdb.connect()

for name in [
    "train_source1.parquet",
    "train_source2.parquet",
    "train_source3.parquet",
]:
    p = CACHE_DIR / name

    n = db.execute(
        f"SELECT COUNT(*) FROM read_parquet('{str(p).replace("\\", "/")}')"
    ).fetchone()[0]

    print(f"{name}: {n:,} rows")

db.close()

# ------------------------------------------------------------
# 5. Create mini ground truth
# ------------------------------------------------------------
MINI_GT = DATA_DIR / "train_ground_truth.tsv"

print("\nCreating mini ground truth...")

wanted = s1_ids
written = 0

with GT_FULL.open("r", encoding="utf-8", newline="") as fin, \
     MINI_GT.open("w", encoding="utf-8", newline="") as fout:

    reader = csv.DictReader(fin, delimiter="\t")

    writer = csv.DictWriter(
        fout,
        fieldnames=["source1_entity_id", "matched_entity_ids"],
        delimiter="\t",
        lineterminator="\n",
    )

    writer.writeheader()

    for row in reader:
        if row["source1_entity_id"] in wanted:
            writer.writerow(row)
            written += 1

            if written == len(wanted):
                break

print(f"✅ Mini GT rows: {written:,}")

# ------------------------------------------------------------
# 6. Import M3 train module
# ------------------------------------------------------------
sys.path.insert(0, str(SRC))

from train import train

print("\n" + "=" * 60)
print("RUNNING M3 SANITY TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# 7. Run baseline M3
# ------------------------------------------------------------
train(
    data_dir=DATA_DIR,
    candidates_path=MINI_CAND,
    output_dir=OUTPUT_DIR,
    models_dir=MODELS_DIR,
    cache_dir=CACHE_DIR,
)

print("\n" + "=" * 60)
print("✅ M3 SANITY TEST FINISHED")
print("=" * 60)

print("\nArtifacts:")
for p in [
    MODELS_DIR / "matcher.pkl",
    MODELS_DIR / "model_config.json",
    OUTPUT_DIR / "baseline_training_results.json",
    OUTPUT_DIR / "baseline_threshold_results.tsv",
]:
    print(("✅" if p.exists() else "❌"), p)

✅ Repository: /content/amazon-ml-challenge-2026
✅ 10K union: /content/union_10k/candidate_pairs_10k.tsv
✅ M2 cache: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m2_cache
✅ Ground truth: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv

Selecting first 1,000 S1 entities...
✅ Created: /content/m3_sanity/candidate_pairs_1000.tsv

Candidate ID counts:
S1: 1000
S2: 72760
S3: 78124

Creating mini M2 cache...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Mini cache created
train_source1.parquet: 1,000 rows
train_source2.parquet: 72,760 rows
train_source3.parquet: 78,124 rows

Creating mini ground truth...
✅ Mini GT rows: 1,000

RUNNING M3 SANITY TRAINING
  Loading sources from M2 Parquet cache...
  LOAD  train_source1.parquet  (1,000 rows, 0.212s)
  LOAD  train_source2.parquet  (72,760 rows, 0.768s)
  LOAD  train_source3.parquet  (78,124 rows, 0.730s)
Loaded  S1=1,000  S2=72,760  S3=78,124  GT=1,000
Candidate rows: 1,000
Labeled pairs : 158,116  (positive=1,652, negative=156,464)
Train S1=800  pairs=133,057
Val   S1=200   pairs=25,059
Model trained.

Threshold sweep results:
 threshold  precision   recall     f0_5
      0.50   0.604619 0.342402 0.540025
      0.55   0.605333 0.340402 0.540142
      0.60   0.610786 0.335402 0.541704
      0.65   0.603274 0.329777 0.534589
      0.70   0.603774 0.325277 0.533828
      0.75   0.599369 0.321944 0.529805
      0.80   0.597452 0.312265 0.524422
      0.85   0.595786 0.304848 0.519645
     

In [12]:
from pathlib import Path
import shutil

src = Path("/content/m3_sanity")

dst = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/m3_sanity"
)

dst.mkdir(parents=True, exist_ok=True)

for f in [
    src / "models/matcher.pkl",
    src / "models/model_config.json",
    src / "output/baseline_training_results.json",
    src / "output/baseline_threshold_results.tsv",
]:
    if f.exists():
        target = dst / f.name
        shutil.copy2(f, target)
        print("✅ Saved:", target)

✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m3_sanity/matcher.pkl
✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m3_sanity/model_config.json
✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m3_sanity/baseline_training_results.json
✅ Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/processed/m3_sanity/baseline_threshold_results.tsv


In [13]:
# ============================================================
# BUILD FILTERED CACHE FOR FULL 10K M3
# ============================================================

from pathlib import Path
import csv
import duckdb
import pandas as pd

UNION = Path("/content/union_10k/candidate_pairs_10k.tsv")

FULL_CACHE = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/processed/m2_cache"
)

GT_FULL = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/ raw/train/train_ground_truth.tsv"
)

BASE = Path("/content/m3_10k")
CACHE_10K = BASE / "cache"
DATA_10K = BASE / "data"

CACHE_10K.mkdir(parents=True, exist_ok=True)
DATA_10K.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Read 10K candidate file and collect all IDs
# ------------------------------------------------------------

print("Reading 10K candidate union...")

s1_ids = set()
s2_ids = set()
s3_ids = set()

with UNION.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        sid = row["source1_entity_id"]
        s1_ids.add(sid)

        value = row.get("candidate_entity_ids", "") or ""

        for cid in value.split(","):
            cid = cid.strip()

            if cid.startswith("S2-"):
                s2_ids.add(cid)
            elif cid.startswith("S3-"):
                s3_ids.add(cid)

print("\nIDs collected:")
print(f"S1 : {len(s1_ids):,}")
print(f"S2 : {len(s2_ids):,}")
print(f"S3 : {len(s3_ids):,}")

# ------------------------------------------------------------
# Filter M2 Parquet cache
# ------------------------------------------------------------

con = duckdb.connect()

con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA preserve_insertion_order=false")

print("\nFiltering S1 cache...")

s1_table = pd.DataFrame({
    "entity_id": list(s1_ids)
})

con.register("wanted_s1", s1_table)

con.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source1.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN wanted_s1 w
            ON s.entity_id = w.entity_id
    )
    TO '{str(CACHE_10K / "train_source1.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

print("✅ S1 cache ready")

# ------------------------------------------------------------

print("Filtering S2 cache...")

s2_table = pd.DataFrame({
    "entity_id": list(s2_ids)
})

con.register("wanted_s2", s2_table)

con.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source2.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN wanted_s2 w
            ON s.entity_id = w.entity_id
    )
    TO '{str(CACHE_10K / "train_source2.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

print("✅ S2 cache ready")

# ------------------------------------------------------------

print("Filtering S3 cache...")

s3_table = pd.DataFrame({
    "entity_id": list(s3_ids)
})

con.register("wanted_s3", s3_table)

con.execute(f"""
    COPY (
        SELECT s.*
        FROM read_parquet(
            '{str(FULL_CACHE / "train_source3.parquet").replace("\\", "/")}'
        ) s
        INNER JOIN wanted_s3 w
            ON s.entity_id = w.entity_id
    )
    TO '{str(CACHE_10K / "train_source3.parquet").replace("\\", "/")}'
    (FORMAT PARQUET)
""")

print("✅ S3 cache ready")

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("10K FILTERED CACHE")
print("=" * 60)

for name in [
    "train_source1.parquet",
    "train_source2.parquet",
    "train_source3.parquet",
]:
    p = CACHE_10K / name

    n = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{str(p).replace(chr(92), "/")}')"
    ).fetchone()[0]

    print(f"{name:25s}: {n:,} rows")

con.close()

print("\n✅ FULL 10K M3 CACHE READY")
print("Location:", CACHE_10K)

Reading 10K candidate union...

IDs collected:
S1 : 10,000
S2 : 960,820
S3 : 978,644

Filtering S1 cache...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ S1 cache ready
Filtering S2 cache...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ S2 cache ready
Filtering S3 cache...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ S3 cache ready

10K FILTERED CACHE
train_source1.parquet    : 10,000 rows
train_source2.parquet    : 960,820 rows
train_source3.parquet    : 978,644 rows

✅ FULL 10K M3 CACHE READY
Location: /content/m3_10k/cache


In [16]:
# ============================================================
# FIX: PUT GROUND TRUTH INTO M3 DATA DIRECTORY
# ============================================================

from pathlib import Path
import shutil

GT_SOURCE = Path(
    "/content/drive/MyDrive/"
    "Amazon ML Challenge 2026/"
    "01_Dataset/ raw/train/train_ground_truth.tsv"
)

DATA_10K = Path("/content/m3_10k/data")
DATA_10K.mkdir(parents=True, exist_ok=True)

GT_TARGET = DATA_10K / "train_ground_truth.tsv"

print("Source:", GT_SOURCE)
print("Target:", GT_TARGET)

if not GT_SOURCE.exists():
    raise FileNotFoundError(f"Ground truth not found: {GT_SOURCE}")

shutil.copy2(GT_SOURCE, GT_TARGET)

print("\n✅ Ground truth copied successfully")
print("Exists:", GT_TARGET.exists())
print("Size:", round(GT_TARGET.stat().st_size / (1024**2), 2), "MB")

Source: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/ raw/train/train_ground_truth.tsv
Target: /content/m3_10k/data/train_ground_truth.tsv

✅ Ground truth copied successfully
Exists: True
Size: 121.13 MB


In [ ]:
train(
    data_dir=DATA_10K,
    candidates_path=CANDIDATES,
    output_dir=OUTPUT,
    models_dir=MODELS,
    cache_dir=CACHE_10K,
)

  Loading sources from M2 Parquet cache...
  LOAD  train_source1.parquet  (10,000 rows, 0.052s)
  LOAD  train_source2.parquet  (960,820 rows, 5.441s)
  LOAD  train_source3.parquet  (978,644 rows, 7.710s)
Loaded  S1=10,000  S2=960,820  S3=978,644  GT=2,206,821
Candidate rows: 10,000
Labeled pairs : 3,130,620  (positive=21,749, negative=3,108,871)
Train S1=7,466  pairs=2,488,250
Val   S1=1,867   pairs=642,370
